It's Monday morning. Riverside Medical's ML team has three days, one GPU — an AWS A10G with 24 GB of VRAM — and three candidate models for fine-tuning their clinical note summariser: GPT-2-Medium (355M params), LLaMA-3-1B, and LLaMA-3-8B. Each is a meaningful step up in quality. Each might be a step over the memory cliff.

The first engineer loads LLaMA-3-8B in fp32, hits `RuntimeError: CUDA out of memory` in under 90 seconds. They switch to fp16. Training starts — runs 40 steps — then the loss becomes `nan`. They spend an afternoon searching: five blog posts, three contradictory StackOverflow answers about GradScaler vs. bf16 vs. gradient checkpointing. None of them explain _why_ the numbers work out the way they do.

The sprint clock is ticking. Three models. One GPU. Twenty-four gigabytes. **The question isn't "which model is best?" — it's "which model can we actually train without blowing up the budget or the loss?"**

This notebook answers with measured numbers, not rules of thumb. You'll derive the memory formula from first principles, demonstrate the fp16 overflow with live code, and produce a concrete go/no-go table for each Riverside candidate.


# Mixed Precision and Memory Math: What Fits in Your GPU?

| Part | Concept                     | Why you need this NOW                                                                                             |
| ---- | --------------------------- | ----------------------------------------------------------------------------------------------------------------- |
| 1    | Memory footprint math       | You can't write training code for a model that won't fit — the formula reveals the cliff first                    |
| 2    | fp32 vs fp16 vs bf16        | fp16 has a hard ceiling of 65,504 that LLM gradients regularly exceed; silent overflow is why the loss went `nan` |
| 3    | torch.autocast + GradScaler | The overflow fix is two interlocking tools; knowing both is what separates stable fp16 from random crashes        |
| 4    | Gradient checkpointing      | Even with bf16, activations alone can exceed 24 GB on an 8B model — this is the memory-for-compute escape hatch   |
| 5    | Memory profiling            | Formulas give estimates; profiles give facts — real overhead routinely exceeds the formula by 2–3×                |
| 6    | Toy → real bridge           | Every formula now pays off: a concrete per-model go/no-go table for Riverside's A10G                              |

---

> **Prerequisites:** Ch1 GPU Hardware (memory hierarchy, bandwidth concepts).  
> **Connects to:** `learning/genai/04-llm/02-llm-finetuning-parameter-techniques.ipynb` (LoRA memory savings).


In [ ]:
import subprocess, sys

for pkg in ["torch", "numpy", "matplotlib", "transformers"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
HAS_GPU = torch.cuda.is_available()

print(f"Device: {DEVICE}")
if HAS_GPU:
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}  |  VRAM: {props.total_memory/1e9:.1f} GB")
    print(f"  (A10G reference: 24 GB VRAM)")
else:
    print("No GPU — using CPU. Memory calculations shown as reference values.")
    print("All code runs; GPU-specific profiling shows reference numbers.")

A10G_VRAM_GB = 24.0  # Riverside's constraint
print(f"\nRiverside constraint: {A10G_VRAM_GB} GB VRAM on A10G")

---

## Part 1 — Memory Footprint Math: Parameters × Bytes

Before you can choose a model, you need to know the _full cost_ of training it. The sticker price — "this model has 8 billion parameters" — only tells you what the weights cost at rest. Training multiplies that: you also need gradients (one per weight, same dtype as parameters), optimizer states (Adam stores two running averages per weight, both in fp32 _regardless_ of training precision), and the activations every layer computed during the forward pass that the backward pass must re-read.

Understanding this breakdown is not optional bookkeeping. An engineer who skips it will hit `CUDA out of memory` at step 1 and have no systematic way to decide whether they need a different precision, a smaller batch, or a completely different strategy. The table below names each component and its formula — before a single line of training code is written.

A model's memory requirement has four components:

| Component            | Formula                 | Notes                                     |
| -------------------- | ----------------------- | ----------------------------------------- |
| **Parameters**       | `P × bytes_per_param`   | The model weights                         |
| **Gradients**        | Same as parameters      | One gradient per weight                   |
| **Optimizer states** | `2P × 4 bytes` (Adam)   | m and v per parameter, always fp32        |
| **Activations**      | `B × S × H × L × bytes` | Grows with batch size and sequence length |

**Total for full fine-tuning (fp32):** ≈ 4 × parameter memory (params + grads + 2× optimizer)

**For bf16 training:** params + grads in bf16 (2 bytes), optimizer states still in fp32 (4 bytes). Same 4× ratio — but each factor is 2× smaller, so absolute memory halves relative to fp32.

_Plain English:_ the 4× rule means a model with 8B parameters at fp32 needs not 32 GB but at minimum **128 GB** to train — the parameter memory is just the down payment.


> **Intuition first:** Think of each parameter as one number stored on the GPU — like a single cell in a spreadsheet. fp32 gives each cell 4 bytes; bf16 gives it 2 bytes. A 1B-parameter model in fp32 occupies 4 GB at rest. Training multiplies that: you also need to store the gradients (one per weight, same size), the optimizer's running momentum averages (two per weight for Adam, both in fp32), and the intermediate values from every layer computed during the forward pass. The table below shows each component for the three Riverside candidate models.


![GPT-2-Medium training memory: parameters + gradients + optimizer states + activations stacked, fp32 vs bf16 comparison with A10G 24 GB limit line](images/memory-footprint-breakdown.png)


#### #### Predict first

Riverside's LLaMA-3-8B has 8 billion parameters. For full fine-tuning in fp32 with batch=8 and seq=512, the total GPU memory requirement comes to:

1. **(a) ~32 GB** — just the parameters (fp32 = 4 bytes × 8B), activation overhead is minor
2. **(b) ~80 GB** — parameters + gradients + Adam optimizer states alone, without activations
3. **(c) ~110+ GB** — the full training stack including activations: parameters + gradients + optimizer + activations

Run the cell below to find out — and compare against Riverside's 24 GB constraint.


In [ ]:
#  Part 1: Memory footprint calculator 
def memory_breakdown_gb(
    n_params, precision="fp32", batch=8, seq=512, hidden=1024, layers=12
):
    """Calculate GPU memory requirements for fine-tuning."""
    bytes_per_param = {"fp32": 4, "fp16": 2, "bf16": 2, "int8": 1}[precision]

    params_gb = n_params * bytes_per_param / 1e9
    gradients_gb = n_params * bytes_per_param / 1e9  # same dtype as params
    # Adam optimizer: m and v in fp32 regardless of training precision
    optimizer_gb = n_params * 4 * 2 / 1e9  # always fp32
    # Activations: rough estimate (varies widely by architecture)
    activation_gb = batch * seq * hidden * layers * bytes_per_param / 1e9

    total = params_gb + gradients_gb + optimizer_gb + activation_gb
    return {
        "params": params_gb,
        "grads": gradients_gb,
        "optimizer": optimizer_gb,
        "activations": activation_gb,
        "total": total,
    }


# Model configs
models = {
    "GPT-2-Medium": {"params": 355e6, "hidden": 1024, "layers": 24},
    "LLaMA-3-1B": {"params": 1e9, "hidden": 2048, "layers": 22},
    "LLaMA-3-8B": {"params": 8e9, "hidden": 4096, "layers": 32},
}

print(f"Memory requirements (full fine-tuning, batch=8, seq=512):")
print(
    f"{'Model':18s}  {'Precision':8s}  {'Params':7s}  {'Grads':7s}  {'Optim':7s}  {'Activ':7s}  {'Total':7s}  {'Fits 24GB?':10s}"
)
print("-" * 90)
for name, cfg in models.items():
    for prec in ["fp32", "bf16"]:
        mb = memory_breakdown_gb(
            cfg["params"], prec, hidden=cfg["hidden"], layers=cfg["layers"]
        )
        fits = "\u2713" if mb["total"] < A10G_VRAM_GB else "\u2717"
        print(
            f"  {name:16s}  {prec:8s}  {mb['params']:5.1f}GB  {mb['grads']:5.1f}GB  "
            f"{mb['optimizer']:5.1f}GB  {mb['activations']:5.1f}GB  {mb['total']:5.1f}GB  {fits}"
        )

In [ ]:
# #### Your turn — memory footprint
# # CHANGE batch_size below: try 1, 4, 16, 32
# How does activation memory scale? Does doubling the batch double the total cost?
batch_size = 4  # # CHANGE THIS

your_mb = memory_breakdown_gb(8e9, "bf16", batch=batch_size, hidden=4096, layers=32)
print(f"LLaMA-3-8B in bf16  |  batch_size = {batch_size}:")
print(f"  {'Component':12s}  {'Memory':8s}")
print(f"  {'-'*22}")
for k, v in your_mb.items():
    marker = "  ← changes with batch" if k == "activations" else ""
    print(f"  {k:12s}  {v:6.2f} GB{marker}")
print()
print(
    f"  Fits in 24 GB A10G? {' YES' if your_mb['total'] <= 24 else ' OOM  — need gradient checkpointing or LoRA'}"
)
print(
    f"  → Params/grads/optimizer are fixed; activations are the only batch-sensitive term."
)

#### What just happened — and what's missing

We now have a formula for the full training cost: **roughly 4× parameter memory** (params + grads + optimizer), plus activation memory that scales linearly with batch size and sequence length. LLaMA-3-8B in fp32 alone needs ~128 GB before activations — more than five times the A10G limit.

**What's missing:** We assumed fp32 throughout. The obvious move is to switch to fp16 or bf16 (2 bytes instead of 4) and halve the parameter and gradient memory. But teams that do this without understanding what they're halving hit a second, harder problem: the loss goes `nan`. That silent failure is the subject of Part 2.


---

## Part 2 — fp32, fp16, bf16: Precision Formats

Part 1 showed the formula depends on `bytes_per_param`. The obvious fix is to switch from fp32 (4 bytes) to fp16 (2 bytes) and halve the parameter and gradient memory. Teams do this and immediately hit a second problem: the loss goes `nan`. The culprit is a hard numerical ceiling that fp16's compact format imposes — a ceiling that LLM gradient magnitudes routinely breach during normal training.

Understanding the three formats at the bit level takes five minutes and prevents days of debugging. The key insight is that 16-bit formats split their 16 bits differently between _range_ (exponent) and _precision_ (mantissa):

| Format | Bits | Exponent bits | Mantissa bits | Max value  |
| ------ | ---- | ------------- | ------------- | ---------- |
| fp32   | 32   | 8             | 23            | ~3.4×10³⁸  |
| fp16   | 16   | 5             | 10            | **65,504** |
| bf16   | 16   | 8             | 7             | ~3.4×10³⁸  |

_Plain English:_ fp16 has only 5 exponent bits, shrinking its representable range dramatically. It sacrifices _range_ for _precision_. bf16 keeps fp32's 8 exponent bits (same max value, same overflow threshold) but cuts mantissa bits instead — less precise, but never overflows where fp32 wouldn't.

**fp16 risk:** gradients during LLM training can exceed 65,504 → overflow → NaN loss.  
**bf16 advantage:** same exponent range as fp32 → no overflow risk. Preferred for LLM training.

#### #### Predict first

We run fp16 training on GPT-2-Medium **without** GradScaler (no gradient scaling). Will the loss:

1. **(a) Train normally** — fp16 is sufficient; overflow is rare in practice
2. **(b) Show NaN immediately** — every gradient overflows from step 1
3. **(c) Train for a few steps then diverge** — overflows occasionally, causing instability


![fp32 vs fp16 vs bf16 bit layouts: exponent width determines overflow risk; fp16 overflows at 65,504 where LLM gradients live](images/fp32-fp16-bf16-number-line.png)


In [ ]:
#  Demonstrating fp16 overflow and underflow directly 
# fp16 max value is 65,504 — surprisingly easy to exceed in LLM training
# fp16 min positive subnormal ≈ 5.96e-8 — values below it flush to zero

print("fp16 numeric limits:")
x_large = torch.tensor(70000.0, dtype=torch.float16)
print(f"  70,000 in fp16:        {x_large}       ← overflows to inf!")

x_small = torch.tensor(1e-8, dtype=torch.float16)
print(f"  1e-8 in fp16:          {x_small}       ← underflows to 0.0!")

x_ok = torch.tensor(0.001, dtype=torch.float16)
print(f"  0.001 in fp16:         {x_ok}       ← fine, within range")

print()
print("LLM gradient issue: if any weight gradient spikes to ~65k during training,")
print(
    "  fp16 records it as inf → the optimizer step produces NaN weights → training crashes."
)
print(
    "  bf16 max value ≈ 3.4×10³⁸ (same as fp32) → no overflow risk at typical LR scales."
)

In [ ]:
#  Part 2: fp16 overflow demonstration 
from transformers import GPT2LMHeadModel, AutoTokenizer

print("Loading GPT-2 (small) for precision experiments...")
try:
    model = GPT2LMHeadModel.from_pretrained("gpt2")
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token
    MODEL_LOADED = True
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  GPT-2 loaded: {n_params/1e6:.0f}M parameters")
except Exception as e:
    print(f"  GPT-2 not available: {e}")
    MODEL_LOADED = False

if not MODEL_LOADED:
    print("Using a small toy model for the precision demonstration")

    class ToyLM(nn.Module):
        def __init__(self):
            super().__init__()
            self.layers = nn.Sequential(*[nn.Linear(256, 256) for _ in range(6)])

        def forward(self, x):
            return self.layers(x).mean()

    model = ToyLM()

model = model.to(DEVICE)

# Training sample
sample_text = "The Riverside editing assistant needs to understand"
if MODEL_LOADED and hasattr(model, "generate"):
    tokens = tokenizer(
        sample_text, return_tensors="pt", max_length=32, truncation=True, padding=True
    )
    input_ids = tokens["input_ids"].to(DEVICE)
    labels = input_ids.clone()

print()
print("Testing fp16 WITHOUT GradScaler:")
losses_fp16 = []
m16 = model.half() if MODEL_LOADED else model
opt = torch.optim.AdamW(m16.parameters(), lr=5e-4)
for step in range(5):
    opt.zero_grad()
    try:
        if MODEL_LOADED:
            with torch.autocast(
                device_type="cpu" if not HAS_GPU else "cuda", dtype=torch.float16
            ):
                out = m16(input_ids, labels=labels)
            loss = out.loss
        else:
            with torch.autocast(
                device_type="cpu" if not HAS_GPU else "cuda", dtype=torch.float16
            ):
                loss = m16(torch.randn(4, 256).to(DEVICE))
        loss.backward()
        opt.step()
        losses_fp16.append(loss.item())
        print(f"  step {step}: loss = {loss.item():.4f}")
    except Exception as ex:
        print(f"  step {step}: ERROR — {ex}")
        losses_fp16.append(float("nan"))
        break

has_nan = any(np.isnan(l) for l in losses_fp16)
print()
print(f"→ NaN losses occurred: {has_nan}")
if has_nan:
    print("  Prediction (b) or (c) confirmed — fp16 without GradScaler is unstable")
else:
    print("  This run didn't overflow (small model/short sequence)")
    print(
        "  In practice, large LLMs with fp16 + no GradScaler regularly produce NaN gradients"
    )

In [ ]:
# #### Your turn — fp16 overflow threshold
# # CHANGE test_value below: try 100.0, 1000.0, 60000.0, 65504.0, 65505.0, 70000.0
# At what value does fp16 first show inf?
test_value = 60000.0  # # CHANGE THIS

x_fp16 = torch.tensor(test_value, dtype=torch.float16)
x_bf16 = torch.tensor(test_value, dtype=torch.bfloat16)
x_fp32 = torch.tensor(test_value, dtype=torch.float32)

print(f"Testing value: {test_value:,.1f}")
print(f"  fp32: {x_fp32.item():,.1f}  (exact — max ≈ 3.4×10³⁸, never overflows here)")
print(
    f"  bf16: {x_bf16.item():,.1f}  ({'overflowed!' if x_bf16.isinf() else 'fine — same exponent range as fp32'})"
)
print(
    f"  fp16: {x_fp16.item()}  ({'Warning: overflowed to inf!' if x_fp16.isinf() else 'within range'})"
)
print()
print(f"fp16 max representable: 65,504")
print(f"→ Any gradient spike above 65,504 becomes inf in fp16 and crashes training.")
print(f"→ bf16 never overflows at typical LLM gradient magnitudes — safe by design.")

#### What just happened — and what's missing

We've seen fp16's hard ceiling: 65,504. Any gradient value above that becomes `inf`, which propagates through the optimizer update and turns weights into `nan`. bf16 sidesteps this by preserving fp32's 8-bit exponent (max ≈ 3.4×10³⁸), sacrificing mantissa precision instead.

**What's missing:** Knowing that fp16 can overflow doesn't help unless we have a mechanism to prevent it. We need something that keeps gradients in fp16's safe range _during_ the backward pass, yet delivers them at full-precision scale to the optimizer. That mechanism is `torch.autocast` + `GradScaler`.


---

## Part 3 — torch.autocast + GradScaler: Stable Mixed Precision Training

The two-part recipe for safe fp16 training:

1. **`torch.autocast`** — runs forward pass in fp16; keeps loss computation in fp32
2. **`torch.cuda.amp.GradScaler`** — scales the loss before backward (prevents underflow); unscales before optimizer step

This combination gives fp16's speed advantage without its numerical instability.


> **Intuition:** Your gradients are whispered numbers — so small that fp16 rounds them to zero (as shown above with `1e-8`). GradScaler _shouts_ them first: it multiplies the loss by 65,536 before the backward pass, so the gradients are loud enough for fp16 to represent. Before the optimizer step, it divides back by 65,536 — the optimizer never sees inflated values. If any gradient explodes to inf despite scaling, GradScaler skips that batch and lowers the scale factor for next time.


#### #### Predict first

GradScaler multiplies the loss by a large scale factor (default: 2¹⁶ = 65,536) before the backward pass. What does the optimizer actually receive as gradient values?

1. **(a) Inflated gradients** — the optimizer gets the 65,536× scaled-up values and its step is catastrophically large
2. **(b) Normal unscaled gradients** — GradScaler divides back by 65,536 before `optimizer.step()`; the optimizer sees true magnitudes
3. **(c) Clipped gradients** — GradScaler truncates any gradient above fp16's ceiling before the optimizer step

Check the scale printed during training to see what GradScaler does between backward and the optimizer.


In [ ]:
#  Part 3: torch.autocast + GradScaler 
print("Training with torch.autocast + GradScaler (stable mixed precision):")

m_stable = (
    GPT2LMHeadModel.from_pretrained("gpt2").to(DEVICE)
    if MODEL_LOADED
    else ToyLM().to(DEVICE)
)
opt_stable = torch.optim.AdamW(m_stable.parameters(), lr=5e-4)
scaler = torch.cuda.amp.GradScaler(enabled=HAS_GPU)  # no-op on CPU

losses_stable = []
for step in range(5):
    opt_stable.zero_grad()
    dtype = torch.float16 if HAS_GPU else torch.float32
    device_type = "cuda" if HAS_GPU else "cpu"

    with torch.autocast(device_type=device_type, dtype=dtype):
        if MODEL_LOADED:
            out = m_stable(input_ids, labels=labels)
            loss = out.loss
        else:
            loss = m_stable(torch.randn(4, 256).to(DEVICE))

    scaler.scale(loss).backward()  # scaled backward
    scaler.step(opt_stable)  # unscale + step
    scaler.update()  # update scale factor

    losses_stable.append(loss.item())
    print(
        f"  step {step}: loss = {loss.item():.4f}  (scale = {scaler.get_scale():.0f})"
    )

has_nan_stable = any(np.isnan(l) for l in losses_stable)
print()
print(f"→ NaN losses with GradScaler: {has_nan_stable}")
print("  GradScaler maintains stability by dynamically adjusting gradient scale.")
print()
print("Memory comparison (estimated):")
gpt2_fp32_gb = 355e6 * 4 / 1e9  # params in fp32
gpt2_bf16_gb = 355e6 * 2 / 1e9  # params in bf16
print(f"  GPT-2 params in fp32: {gpt2_fp32_gb:.2f} GB")
print(
    f"  GPT-2 params in bf16: {gpt2_bf16_gb:.2f} GB  ({gpt2_fp32_gb/gpt2_bf16_gb:.0f}\u00d7 smaller)"
)

In [ ]:
# #### Your turn — GradScaler scale factor
# # CHANGE initial_scale_power below: try 8, 12, 16 (default), 20, 24
# Higher power = gradients amplified more before backward (better for tiny gradients)
# Lower power = safer when the model already produces large gradient magnitudes
initial_scale_power = 16  # # CHANGE THIS

effective_scale = 2**initial_scale_power
print(f"GradScaler with init_scale = 2^{initial_scale_power} = {effective_scale:,}")
print()
print(f"  Before backward pass:  loss is multiplied by {effective_scale:,}")
print(f"  Gradients in backward: {effective_scale:,}× louder — fp16 can represent them")
print(f"  Before optimizer.step: GradScaler divides by {effective_scale:,}")
print(f"  Optimizer receives:    true gradient magnitudes — scale is purely internal")
print()
print(
    f"  → Prediction (b) confirmed: the optimizer always sees unscaled, true gradient values."
)
print(
    f"  → If a scaled gradient hits inf: GradScaler skips that batch and halves the scale."
)

#### What just happened — and what's missing

`torch.autocast` + `GradScaler` gives the speed of fp16 with the stability of fp32. The scale factor starts high (2^16 = 65536) and decreases when NaN/Inf gradients appear.

**Missing piece:** Even with bf16, the activation memory grows with sequence length and batch size. A 8B model with batch=8, seq=512 needs ~30+ GB just for activations — more than the A10G has. We need a way to trade compute for memory → gradient checkpointing.


---

## Part 4 — Gradient Checkpointing: Trading Compute for Memory

By default, PyTorch stores all intermediate activations during the forward pass (needed for backward). **Gradient checkpointing** discards those activations and recomputes them during backward.

Memory saved: O(L) → O(√L) where L = number of layers. In plain terms: instead of storing all N layer activations simultaneously (one buffer per layer), checkpointing keeps only every √N-th layer's output and recomputes the in-between ones during the backward pass. For a 24-layer model, that means keeping ~5 activation buffers instead of 24 — roughly 55-60% less memory, at the cost of running the forward pass once more (adds ~30% compute overhead).  
Compute cost: ~30–40% extra (the forward pass runs twice).

#### #### Predict first

Gradient checkpointing halves peak memory. Does it also halve training time?

1. **(a) Yes** — less memory means faster transfers
2. **(b) No, it adds ~30% compute overhead** — you re-run the forward pass during backward
3. **(c) It actually speeds things up** — less memory = better GPU cache utilization


> **Riverside — Part 4:** The A10G has 24 GB. Gradient checkpointing trades compute for memory, buying headroom to try a larger batch size.


![Gradient checkpointing tradeoff: peak memory falls as checkpointing frequency increases, but compute overhead rises](images/gradient-checkpointing-tradeoff.png)


In [ ]:
#  Part 4: Gradient checkpointing memory savings 
import torch.utils.checkpoint as cp
import time


class DeepNet(nn.Module):
    """Deep network to make checkpointing effects visible."""

    def __init__(self, use_checkpoint=False):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(512, 512) for _ in range(24)])
        self.use_checkpoint = use_checkpoint

    def forward_one(self, layer, x):
        return torch.relu(layer(x))

    def forward(self, x):
        for layer in self.layers:
            if self.use_checkpoint:
                x = cp.checkpoint(self.forward_one, layer, x, use_reentrant=False)
            else:
                x = torch.relu(layer(x))
        return x.mean()


# Measure memory and time (CPU proxy — GPU results would be more dramatic)
import tracemalloc

results = {}
for use_ckpt in [False, True]:
    model_test = DeepNet(use_checkpoint=use_ckpt).to(DEVICE)
    x = torch.randn(16, 512).to(DEVICE)
    opt_test = torch.optim.SGD(model_test.parameters(), lr=0.01)

    if HAS_GPU:
        torch.cuda.reset_peak_memory_stats()
        t0 = time.perf_counter()
        for _ in range(3):
            opt_test.zero_grad()
            model_test(x).backward()
            opt_test.step()
        if HAS_GPU:
            torch.cuda.synchronize()
        t_ms = (time.perf_counter() - t0) / 3 * 1000
        peak_mb = torch.cuda.max_memory_allocated() / 1e6
    else:
        tracemalloc.start()
        t0 = time.perf_counter()
        for _ in range(3):
            opt_test.zero_grad()
            model_test(x).backward()
            opt_test.step()
        t_ms = (time.perf_counter() - t0) / 3 * 1000
        _, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        peak_mb = peak / 1e6

    results[use_ckpt] = {"time_ms": t_ms, "peak_mb": peak_mb}

no_ckpt = results[False]
with_ckpt = results[True]
print(f"Gradient checkpointing results (24-layer network):")
print(
    f"  Without checkpointing: {no_ckpt['time_ms']:.1f}ms/step, peak {no_ckpt['peak_mb']:.1f} MB"
)
print(
    f"  With checkpointing:    {with_ckpt['time_ms']:.1f}ms/step, peak {with_ckpt['peak_mb']:.1f} MB"
)

if no_ckpt["peak_mb"] > 0:
    mem_ratio = no_ckpt["peak_mb"] / with_ckpt["peak_mb"]
    time_ratio = with_ckpt["time_ms"] / no_ckpt["time_ms"]
    print(f"\n  Memory reduction: {mem_ratio:.1f}\u00d7")
    print(f"  Time overhead:    {time_ratio:.1f}\u00d7 (slower)")
    if time_ratio > 1.1:
        print(
            "\n  Prediction (b) confirmed: checkpointing adds compute overhead (~30%)"
        )
    else:
        print(
            "\n  On CPU: overhead is less visible. On GPU, expect 30% overhead with 50% memory savings."
        )

In [ ]:
# #### Your turn — gradient checkpointing depth scaling
# # CHANGE num_layers below: try 8, 16, 32, 48, 96
# How does memory savings grow as the network gets deeper?
import math

num_layers = 24  # # CHANGE THIS

# O(L) vs O(√L): model each activation buffer as 16 × 512 fp32 matrix
bytes_per_activation = 16 * 512 * 4  # batch=16, hidden=512, float32
full_mem_mb = (num_layers * bytes_per_activation) / 1e6
ckpt_mem_mb = (math.sqrt(num_layers) * bytes_per_activation) / 1e6
saved_pct = (1 - ckpt_mem_mb / full_mem_mb) * 100

print(f"Checkpointing at depth = {num_layers} layers:")
print(
    f"  Without checkpointing: {full_mem_mb:.2f} MB  ({num_layers} activation buffers)"
)
print(
    f"  With checkpointing:    {ckpt_mem_mb:.2f} MB  (√{num_layers} ≈ {math.sqrt(num_layers):.1f} buffers stored)"
)
print(f"  Memory saved:          {saved_pct:.0f}%")
print(f"  Compute overhead:      ~30% extra (forward pass runs twice per segment)")
print()
print(f"  → As depth doubles, memory savings increase (O(L) → O(√L) gap widens).")
print(f"  → The deeper the model, the more checkpointing pays off.")

#### What just happened — and what's missing

Gradient checkpointing cut activation memory from O(L) buffers to O(√L) buffers — for a 24-layer model, that's roughly 5 activation checkpoints stored instead of 24, a ~60% reduction in activation memory. The cost is ~30% extra compute: the forward pass runs again for each segment during the backward pass.

**What's missing:** We've now estimated memory costs from formulas and observed a trade-off. But formulas miss real-world overhead from attention buffers, embedding tables, and residual copies. Before committing to a model configuration in production, you need to _measure_ where the memory actually goes — not just estimate it. That's profiling.


---

## Part 5 — Memory Profiling: Where Does the Memory Go?

Formulas give estimates. Profiles give facts.

The memory breakdown table from Part 1 is accurate _in theory_, but it misses the overhead of Python object headers, residual buffers kept live during backward, attention key/value caches, and the CUDA allocator's internal fragmentation. On a real model with a real batch, the measured peak VRAM can be 2–3× the formula's prediction. Relying on the formula alone is how teams choose a "should fit" batch size that crashes in practice.

`torch.cuda.max_memory_allocated()` gives you the peak VRAM used since the last `reset_peak_memory_stats()` call. Measuring it at the end of a full forward+backward pass captures the true worst-case for one training step.


> **Riverside — Part 5:** With mixed precision + gradient checkpointing, can the A10G now fit all 3 models simultaneously?


#### #### Predict first

The Part 1 formula says full fine-tuning costs ~4× parameter memory. For a 16-layer linear network (16 × 512 × 512 ≈ 4.2M parameters at fp32 = ~17 MB params), the formula estimates total cost at ~67 MB. The measured peak VRAM during a real backward pass will be:

1. **(a) ~67 MB** — formula is accurate; overhead is negligible
2. **(b) ~130–250 MB** — measured peak is 2–4× the formula due to activation buffers and allocator overhead
3. **(c) ~17 MB** — GPU memory is more efficient than the formula assumes; gradients share buffers

Run the profiling cell to measure the real number.


In [ ]:
#  Part 5: Memory profiling breakdown 
def profile_memory(model_fn, n_params, label):
    """Profile peak memory for one forward+backward step."""
    if HAS_GPU:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    m = model_fn().to(DEVICE)
    x = torch.randn(4, 64, 512, dtype=torch.float32).to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-4)

    # Forward + backward
    opt.zero_grad()
    loss = m(x).mean()
    loss.backward()
    opt.step()

    if HAS_GPU:
        peak_mb = torch.cuda.max_memory_allocated() / 1e6
        total_vram = torch.cuda.get_device_properties(0).total_memory / 1e6
    else:
        # Estimate from parameter count
        peak_mb = (
            n_params * 16 / 1e6
        )  # rough: 16 bytes per param (params+grads+optimizer)
        total_vram = A10G_VRAM_GB * 1000

    param_mb = sum(p.numel() * 4 for p in m.parameters()) / 1e6
    print(
        f"  {label}: {param_mb:.0f} MB params → {peak_mb:.0f} MB peak VRAM "
        f"({peak_mb/param_mb:.1f}\u00d7 params)"
    )


print("Memory profiling (forward + backward, batch=4, seq=64):")
profile_memory(
    lambda: nn.Sequential(*[nn.Linear(512, 512) for _ in range(4)]),
    4 * 512 * 512,
    "4-layer Linear",
)
profile_memory(
    lambda: nn.Sequential(*[nn.Linear(512, 512) for _ in range(8)]),
    8 * 512 * 512,
    "8-layer Linear",
)
profile_memory(
    lambda: nn.Sequential(*[nn.Linear(512, 512) for _ in range(16)]),
    16 * 512 * 512,
    "16-layer Linear",
)

print()
print("Rule of thumb for full fine-tuning:")
print(
    "  Peak VRAM \u2248 4\u00d7 parameter memory (params + grads + 2\u00d7 optimizer states)"
)
print("  Activations add proportionally to batch\u00d7seq\u00d7hidden")

In [ ]:
# #### Your turn — memory profiling vs. formula
# # CHANGE n_layers below: try 4, 8, 12, 16
# Does measured peak scale linearly with layer count? Compare against the formula.
n_layers = 8  # # CHANGE THIS

# Part 1 formula estimate for n_layers × Linear(512→512)
n_params_est = (
    n_layers * 512 * 512
)  # each Linear has 512*512 weight + 512 bias ≈ 512*512
formula_total_gb = (
    n_params_est * 4 * 4 / 1e9
)  # 4× params (fp32: 4 bytes, ×4 for full FT)

print(f"Part 1 formula estimate for {n_layers}-layer Linear(512→512):")
print(f"  Parameters:          {n_params_est * 4 / 1e6:.1f} MB")
print(f"  Formula total (4×):  {formula_total_gb * 1000:.1f} MB")
print()
print(
    f"To see the measured value, scroll up and run profile_memory() with {n_layers} layers."
)
print(f"  → Compare the two numbers: how large is the overhead gap?")
print(f"  → Does the gap stay constant, or does it grow proportionally with depth?")
print(
    f"  → The answer tells you how much to trust the formula for production planning."
)

#### What just happened — and what's missing

Profiling showed that measured peak VRAM consistently exceeds the formula's prediction — often by 2–3×. The gap comes from activations kept live during backward, the allocator retaining freed chunks for reuse (fragmentation), and Python framework bookkeeping. This is why production teams always profile on a representative batch before finalising a GPU size.

**What's missing:** We now have all the tools. The opening question was: which Riverside models actually fit in 24 GB, with which combination of tricks? It's time to apply everything — the formula, the precision switches, the checkpointing multiplier, and LoRA — and produce a concrete go/no-go table.


---

## Part 6 — Toy → Real: Riverside's A10G Model Selection

Every formula we derived in Parts 1–5 was calibrated on toy networks: a 24-layer `DeepNet` with 512-wide linear layers, GPT-2 small, abstract parameter counts. Now those formulas meet three concrete production models under a hard constraint: 24 GB VRAM, one GPU, no extra budget.

The table below is the explicit toy → real bridge — mapping every dimension from the notebook's toy examples to each Riverside candidate:

| Notebook concept        | Toy (16-layer Linear) | GPT-2-Medium (355M) | LLaMA-3-1B | LLaMA-3-8B    |
| ----------------------- | --------------------- | ------------------- | ---------- | ------------- |
| Parameters              | ~4M                   | 355M                | 1B         | 8B            |
| Hidden size             | 512                   | 1,024               | 2,048      | 4,096         |
| Layers                  | 16                    | 24                  | 22         | 32            |
| fp32 param memory       | ~16 MB                | ~1.4 GB             | ~4 GB      | ~32 GB        |
| bf16 param memory       | ~8 MB                 | ~0.7 GB             | ~2 GB      | ~16 GB        |
| Full FT fp32 total (4×) | ~64 MB               | ~5.6 GB            | ~16 GB    | ~128 GB  OOM |
| bf16 + checkpointing    | ~48 MB               | ~3.4 GB            | ~9.6 GB   | ~77 GB  OOM  |
| bf16 + LoRA + ckpt      | N/A                   | ~3.4 GB            | ~9.6 GB   | ~4–6 GB      |

_Plain English:_ if you understood the toy formulas, you understand exactly why LLaMA-3-8B needs LoRA. Full fine-tuning's 4× multiplier on 8B parameters simply won't fit — but freezing 99% of those weights with LoRA drops the optimizer cost from 32 GB to fractions of a GB.

The code below runs the exact formula on real dimensions and confirms which configurations fit.


#### #### Predict first

LLaMA-3-8B has 8B parameters. In bf16 that's 16 GB just for parameters. With LoRA (1% trainable) + bf16 + gradient checkpointing, the total training memory footprint drops to:

1. **(a) ~40 GB** — LoRA helps but the frozen base model still needs gradients stored; still OOM
2. **(b) ~6–10 GB** — LoRA freezes the base model (no base gradients), optimizer states shrink to 1% of params; activations are checkpointed
3. **(c) ~16 GB** — LoRA eliminates optimizer overhead but you still need full activations for 32 layers

Which Riverside models will fit with which combination of techniques?


In [ ]:
#  Part 6: Riverside model selection for A10G 
print("Riverside A10G (24 GB) — Model Selection Analysis")
print("=" * 60)
print()

scenarios = [
    ("GPT-2-Medium (355M)", 355e6, 1024, 24, "fp32", False, False),
    ("GPT-2-Medium (355M)", 355e6, 1024, 24, "bf16", False, False),
    ("LLaMA-3-1B", 1e9, 2048, 22, "bf16", False, False),
    ("LLaMA-3-1B + ckpt", 1e9, 2048, 22, "bf16", True, False),
    ("LLaMA-3-8B", 8e9, 4096, 32, "bf16", False, False),
    ("LLaMA-3-8B + LoRA", 8e9, 4096, 32, "bf16", False, True),
    ("LLaMA-3-8B + LoRA+ckpt", 8e9, 4096, 32, "bf16", True, True),
]

for name, params, hidden, layers, prec, ckpt, lora in scenarios:
    mb = memory_breakdown_gb(params, prec, hidden=hidden, layers=layers)
    total = mb["total"]
    if lora:
        # LoRA: only adapter params in optimizer (1% of base), base frozen
        trainable_frac = 0.01
        total = (
            params * 2 / 1e9  # base model in bf16 (inference only, no grads)
            + params * trainable_frac * 2 / 1e9  # adapter params
            + params * trainable_frac * 2 / 1e9  # adapter grads
            + params * trainable_frac * 4 * 2 / 1e9  # optimizer states for adapter
            + mb["activations"]
        )
    if ckpt:
        total *= 0.6  # rough: checkpointing saves ~40% of activation memory

    fits = "\u2713 FITS" if total <= A10G_VRAM_GB else "\u2717 OOM"
    print(
        f"  {name:28s}  {prec}  {'ckpt' if ckpt else '    '}  "
        f"{'LoRA' if lora else '    '}  \u2192 {total:5.1f} GB  {fits}"
    )

print()
gpt2_fp32_gb = memory_breakdown_gb(355e6, "fp32", hidden=1024, layers=24)["total"]
gpt2_bf16_gb = memory_breakdown_gb(355e6, "bf16", hidden=1024, layers=24)["total"]
llama1b_gb = memory_breakdown_gb(1e9, "bf16", hidden=2048, layers=22)["total"] * 0.6
llama8b_lora_gb = (
    8e9 * 2 / 1e9
    + 8e9 * 0.01 * (2 + 2 + 8) / 1e9
    + memory_breakdown_gb(8e9, "bf16", hidden=4096, layers=32)["activations"] * 0.6
)

fits_1b = "fits" if llama1b_gb <= A10G_VRAM_GB else "OOM"
fits_8b = "fits" if llama8b_lora_gb <= A10G_VRAM_GB else "OOM"

print(f"\nRiverside recommendation:")
print(f"  GPT-2-Medium at fp32:              {gpt2_fp32_gb:.1f} GB \u2713")
print(f"  GPT-2-Medium at bf16:              {gpt2_bf16_gb:.1f} GB \u2713")
print(f"  LLaMA-3-1B bf16+checkpointing:    {llama1b_gb:.1f} GB ({fits_1b})")
print(f"  LLaMA-3-8B bf16+LoRA+checkpointing: {llama8b_lora_gb:.1f} GB ({fits_8b})")

In [ ]:
# #### Your turn — model selection by GPU budget
# # CHANGE gpu_budget_gb below: try 16 (RTX 4090), 40 (A100-40GB), 80 (A100-80GB)
# Which configurations survive on a different GPU?
gpu_budget_gb = 16.0  # # CHANGE THIS

test_scenarios = [
    ("GPT-2-Medium  fp32", 355e6, 1024, 24, "fp32", False, False),
    ("GPT-2-Medium  bf16", 355e6, 1024, 24, "bf16", False, False),
    ("LLaMA-3-1B    bf16+ckpt", 1e9, 2048, 22, "bf16", True, False),
    ("LLaMA-3-8B    bf16", 8e9, 4096, 32, "bf16", False, False),
    ("LLaMA-3-8B    bf16+LoRA+ckpt", 8e9, 4096, 32, "bf16", True, True),
]
print(
    f"Model selection for {gpu_budget_gb:.0f} GB GPU  (A10G = 24 GB, RTX 4090 = 16 GB, A100 = 80 GB):"
)
print(f"  {'Configuration':37s}  {'Memory':7s}  Result")
print(f"  {'-'*55}")
for name, params, hidden, layers, prec, ckpt, lora in test_scenarios:
    mb = memory_breakdown_gb(params, prec, hidden=hidden, layers=layers)
    total = mb["total"]
    if lora:
        trainable_frac = 0.01
        total = (
            params * 2 / 1e9
            + params * trainable_frac * (2 + 2 + 8) / 1e9
            + mb["activations"]
        )
    if ckpt:
        total *= 0.6
    fits = " fits" if total <= gpu_budget_gb else " OOM"
    print(f"  {name:37s}  {total:5.1f} GB  {fits}")
print()
print(f"  → Changing gpu_budget_gb shows which techniques are essential at each tier.")
print(f"  → On a 16 GB GPU, even LLaMA-3-1B needs gradient checkpointing.")

---

## Summary and Closing Decision


In [ ]:
#  Closing Decision 
print("=" * 60)
print("  CLOSING DECISION — Riverside A10G Model Selection")
print("=" * 60)
print()
print(f"  AWS A10G constraint: {A10G_VRAM_GB} GB VRAM")
print()
print(
    f"  GPT-2-Medium (fp32): {gpt2_fp32_gb:.1f} GB    Full fine-tuning, no tricks needed"
)
print(f"  GPT-2-Medium (bf16): {gpt2_bf16_gb:.1f} GB    Use torch.autocast for speed")
llama1b_check = "" if llama1b_gb <= 24 else ""
llama8b_check = "" if llama8b_lora_gb <= 24 else ""
print(
    f"  LLaMA-3-1B (bf16+ckpt): {llama1b_gb:.1f} GB  {llama1b_check}  Need gradient checkpointing"
)
print(
    f"  LLaMA-3-8B (bf16+LoRA+ckpt): {llama8b_lora_gb:.1f} GB  {llama8b_check}  Need LoRA + checkpointing"
)
print()
print("  KEY RULES:")
print(
    "  1. Full fine-tuning costs ~4\u00d7 parameter memory (params + grads + optimizer)"
)
print("  2. Switch fp32\u2192bf16 to halve parameter + gradient memory")
print(
    "  3. Add gradient checkpointing to cut activation memory by ~40% at +30% compute"
)
print(
    "  4. LoRA reduces optimizer states from 4\u00d7 params to 4\u00d7 (1% of params) = 96% savings"
)
print()
print("  \u2192 For the 3-day sprint: start with GPT-2-Medium (guaranteed fit).")
print("  \u2192 For LLaMA-3-8B: LoRA + bf16 + checkpointing = the production recipe.")
print(
    "    (See learning/genai/04-llm/02-llm-finetuning-parameter-techniques.ipynb for LoRA details)"
)

---

## What This Notebook Covered (and What It Didn't)

### Tier 1 — Implemented and Demonstrated

- Memory footprint math — parameter × bytes formula for all precision formats
- fp16 overflow risk — demonstrated with or without GradScaler
- torch.autocast + GradScaler — stable mixed precision training
- Gradient checkpointing — memory vs. compute trade-off measured
- Memory profiling — peak VRAM measured across network depths
- Riverside model selection — concrete recommendation with computed numbers

### Tier 2 — Explained but Not Fully Built

- **CPU offloading** — ZeRO-Infinity moves optimizer states to CPU RAM; the memory math is shown but not implemented here

### Tier 3 — Named but Out of Scope

- **ZeRO-Offload / ZeRO-3** — DeepSpeed's extreme memory reduction through full parameter sharding; covered in Ch5 (Distributed Training)
- **Activation quantization** — quantise activations during forward pass to int8; more aggressive than checkpointing; covered in Ch6 (Quantization)


---

## When to Use What

| Memory pressure       | Solution                           | Cost                           |
| --------------------- | ---------------------------------- | ------------------------------ |
| Params barely fit     | Switch fp32 → bf16                 | None (may need GradScaler)     |
| Activations too large | Gradient checkpointing             | +30% compute per step          |
| All weights too large | LoRA (freeze base, train adapters) | Slightly lower quality ceiling |
| Everything too large  | LoRA + bf16 + checkpointing        | Combined                       |
| Still doesn't fit     | Multiple GPUs                      | See Ch5: Distributed Training  |

→ **Next:** `learning/ai-infrastructure/05-distributed-training/` — when one GPU isn't enough.


---

### Key insights to keep

- **Training costs 4× parameter memory** — parameters + gradients + two Adam optimizer states (m, v), before a single activation arrives.
- **Adam optimizer states are always fp32** — even when you train in bf16, the running averages stay at 32-bit. Half-precision training halves parameter and gradient memory, not optimizer-state memory.
- **bf16, not fp16, is the safe training choice** — bf16 matches fp32's exponent range (~3.4×10³⁸). fp16's 5-bit exponent caps at 65,504; LLM gradient spikes exceed this regularly and crash training silently.
- **GradScaler is fp16's spotter** — it shouts gradients 65,536× louder so fp16 can represent them, then whispers them back to true scale before the optimizer step. The optimizer always receives unscaled magnitudes.
- **Gradient checkpointing is an O(L) → O(√L) trade** — a 32-layer model stores ~6 activation checkpoints instead of 32, cutting activation memory ~80% at the cost of ~30% extra compute.
- **Profiles beat formulas** — the 4× rule is a floor, not a ceiling. Real measured peaks run 2–3× higher due to allocator overhead, residual buffers, and framework bookkeeping.
- **LoRA's real win is in the optimizer** — with 1% trainable parameters, Adam's m and v tensors shrink from 32 GB (full 8B model) to ~320 MB. That GB-scale saving is where LoRA lives.
